# 05. Évaluation Comparative & Test de Diebold-Mariano

**Projet :** Prévision de l'or (XAU/USD) — *Advanced Time Series* (ENSAI, 3A)

### Objectifs :
1. Rassembler les prédictions out-of-sample de tous les modèles :
   - Benchmark naïf (Random Walk)
   - ARIMA / ARMA
   - Modèles de volatilité (GARCH)
   - Modèle Machine Learning
2. Calculer les métriques standard : MAE, RMSE, MAPE, Directional Accuracy (Hit Ratio).
3. Exécuter le **test statistique de Diebold-Mariano** pour valider formellement la supériorité prédictive par rapport au benchmark.
4. Synthèse et interprétation des résultats pour la soutenance / rapport final.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats

DATA_DIR = Path("../data")
FIG_DIR = Path("../figures")

def diebold_mariano_test(y_true, y_pred1, y_pred2, h=1, loss_type="squared"):
    """
    Calcule la statistique du test de Diebold-Mariano entre deux séries de prévisions.
    H0 : Les deux modèles ont la même précision prédictive (E[d_t] = 0).
    H1 : Les précisions prédictives diffèrent.
    """
    e1 = y_true - y_pred1
    e2 = y_true - y_pred2
    
    if loss_type == "squared":
        d = e1**2 - e2**2
    elif loss_type == "absolute":
        d = np.abs(e1) - np.abs(e2)
    else:
        raise ValueError("loss_type doit être 'squared' ou 'absolute'")
        
    d_mean = np.mean(d)
    T = len(d)
    
    # Variance autocovariante (Harvey, Leybourne & Newbold correction)
    gamma0 = np.var(d, ddof=0)
    dm_stat = d_mean / np.sqrt(gamma0 / T)
    p_val = 2 * (1 - stats.norm.cdf(np.abs(dm_stat)))
    
    return dm_stat, p_val

print("Fonction Diebold-Mariano prête.")